In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

%matplotlib inline
from io import BytesIO

from datasets import load_dataset
from PIL import Image
from sklearn import metrics
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

I0000 00:00:1774800110.236362    4181 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/tiberiu/Documents/Dizertatie/deforge-ela/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    CACHE_DIR = None

    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

In [3]:
if IN_COLAB:
    print(f'Loading dataset. Colab detected: using Drive cache at {CACHE_DIR}')
else:
    print('Loading dataset. Local environment detected: using default HF cache.')

train_data = load_dataset(
    'TheKernel01/AIGC-Detection-Benchmark',
    split='train',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)
validation_data = load_dataset(
    'TheKernel01/AIGC-Detection-Benchmark',
    split='validation',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)
test_data = load_dataset(
    'TheKernel01/AIGC-Detection-Benchmark',
    split='test',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

Loading dataset. Local environment detected: using default HF cache.


Using the latest cached version of the dataset since TheKernel01/AIGC-Detection-Benchmark couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/tiberiu/.cache/huggingface/datasets/TheKernel01___aigc-detection-benchmark/default/0.0.0/350a669c65e291dce6e438f6ad95c395f9672e44 (last modified on Sat Mar 28 18:24:36 2026).
Using the latest cached version of the dataset since TheKernel01/AIGC-Detection-Benchmark couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/tiberiu/.cache/huggingface/datasets/TheKernel01___aigc-detection-benchmark/default/0.0.0/350a669c65e291dce6e438f6ad95c395f9672e44 (last modified on Sat Mar 28 18:24:36 2026).
Using the latest cached version of the dataset since TheKernel01/AIGC-Detection-Benchmark couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/tiberiu/.cache/huggingface/datasets/TheKernel01___aigc-dete

In [4]:
BATCH_SIZE = 128
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 2

In [5]:
def ela_from_pil(pil_image: Image.Image, scale=(224, 224), quality=90) -> np.ndarray:
    """
    Performs Error Level Analysis (ELA) on a PIL image and returns a 3-channel RGB result.

    Args:
        pil_image (PIL.Image.Image): Input PIL image.
        scale (tuple): Resize dimensions (width, height).
        quality (int): JPEG quality for recompression.

    Returns:
        np.ndarray: 3-channel ELA image in RGB format (uint8).
    """
    image = pil_image.convert('RGB').resize(scale)

    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)

    compressed = Image.open(buffer)

    diff = np.abs(
        np.array(image, dtype=np.int16) - np.array(compressed, dtype=np.int16)
    )
    diff = np.clip(diff * 10, 0, 255).astype(np.uint8)

    return diff


# %%
def hf_dataset_to_tf_dataset(
    hf_dataset, batch_size: int, shuffle: bool = False
) -> tf.data.Dataset:
    """
    Converts a HuggingFace dataset split to a tf.data.Dataset,
    applying ELA preprocessing and normalisation on the fly.

    The HF dataset provides:
        - 'image'  : PIL.Image
        - 'label'  : int  (0 = real, 1 = fake)

    Args:
        hf_dataset: A HuggingFace dataset split.
        batch_size: Number of samples per batch.
        shuffle:    Whether to shuffle the dataset.

    Returns:
        A batched tf.data.Dataset yielding (image_tensor, one_hot_label) pairs.
    """

    def generator():
        for sample in hf_dataset:
            ela_img = ela_from_pil(sample['image'], scale=IMAGE_SIZE)
            # Normalise to [0, 1]
            x = ela_img.astype(np.float32) / 255.0
            # One-hot encode the binary label
            y = tf.keras.utils.to_categorical(sample['label'], num_classes=NUM_CLASSES)
            yield x, y

    output_signature = (
        tf.TensorSpec(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(generator, output_signature=output_signature)

    if shuffle:
        ds = ds.shuffle(buffer_size=1024)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [6]:
print('Building TensorFlow datasets (ELA will be applied on the fly)...')
train_generator = hf_dataset_to_tf_dataset(train_data, BATCH_SIZE, shuffle=True)
validation_generator = hf_dataset_to_tf_dataset(
    validation_data, BATCH_SIZE, shuffle=False
)
test_generator = hf_dataset_to_tf_dataset(test_data, BATCH_SIZE, shuffle=False)
print('Done.')

Building TensorFlow datasets (ELA will be applied on the fly)...


W0000 00:00:1774800148.957379    4181 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Done.


In [7]:
def build_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation='relu'),
            MaxPooling2D((2, 2)),
            Conv2D(128, (3, 3), activation='relu'),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(NUM_CLASSES, activation='softmax'),
        ]
    )
    optimizer = Adam(learning_rate=0.0001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model


model = build_model()
model.summary()

/home/tiberiu/Documents/Dizertatie/deforge-ela/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1774800149.316426    4181 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1774800149.326355    4181 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1774800149.333480    4181 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,218 (42.61 MB)

 Trainable params: 11,169,218 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
plot_model(
    model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True
)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [9]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10,
)

Epoch 1/10


W0000 00:00:1774800152.402757    4181 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
I0000 00:00:1774800152.882562    4885 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1774800162.881585    4885 shuffle_dataset_op.cc:453] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 1000 of 1024
I0000 00:00:1774800163.121577    4885 shuffle_dataset_op.cc:483] Shuffle buffer filled.
W0000 00:00:1774800164.331495    4885 cpu_allocator_impl.cc:82] Allocation of 77070336 exceeds 10% of free system memory.


KeyboardInterrupt: 

In [ ]:
if not os.path.exists('models'):
    os.makedirs('models', exist_ok=True)

model.save(os.path.join('models', 'deforge_ela.keras'))

# Plotting Accuracy and Loss Graph:

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'val'])

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'val'])
plt.show()

# Confusion Matrix:

In [ ]:
print('Generating predictions on test set...')
y_pred_list, y_test_list = [], []

for x_batch, y_batch in test_generator:
    preds = model.predict(x_batch, verbose=0)
    y_pred_list.append(np.argmax(preds, axis=1))
    y_test_list.append(np.argmax(y_batch.numpy(), axis=1))

y_pred_labels = np.concatenate(y_pred_list)
y_test = np.concatenate(y_test_list)

In [ ]:
Fake = False
Real = True
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=metrics.confusion_matrix(y_test, y_pred_labels),
    display_labels=[Fake, Real],
)
cm_display.plot()
plt.show()

# ROC AUC Score, Precision Score and Test Accuracy:

In [ ]:
print('ROC AUC Score:', metrics.roc_auc_score(y_test, y_pred_labels))
print('AP Score:', metrics.average_precision_score(y_test, y_pred_labels))
print(metrics.classification_report(y_test, y_pred_labels))

In [ ]:
_, accu, _, _ = model.evaluate(test_generator)
print('Final Test Accuracy = {:.3f}'.format(accu * 100))

# Testing for a random image:

In [ ]:
sample = test_data[0]
ela_image = ela_from_pil(sample['image'], IMAGE_SIZE)
p1 = ela_image.astype(np.float32) / 255.0
p1 = np.expand_dims(p1, axis=0)

op = np.argmax(model.predict(p1), axis=-1)
print(op)
if op == [0]:
    print('Real Image')
else:
    print('Fake Image')